In [1]:
import torch
import pandas as pd
import numpy as np
from torchmetrics import Accuracy
from algorithms.pso import PSO
from algorithms.ga import GA
from algorithms.cmaes import CMAES
from algorithms.de import DE
import matplotlib.pyplot as plt

from algorithms.minimize import minimize

In [2]:
np.random.seed(42)

In [3]:
device = "cpu" 

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda" 

device

'cuda'

In [4]:
data = pd.read_csv("Data/winequality-red.csv", sep=";")

X_np = data[data.columns[:-1]].values
y_np = data[data.columns[-1]].values
y_np = y_np.astype("float") + np.random.lognormal(size=y_np.shape[0])
# y_np = np.random.lognormal(size=y_np.shape[0])

In [5]:
X = torch.tensor(X_np, dtype=torch.float32).to(device)
y = torch.tensor(y_np, dtype=torch.float32).to(device) 

In [6]:
loss = torch.nn.MSELoss()

In [7]:
def MLP(theta, x, h_units):
        """tiny MLP"""
        
        _, n_features = x.shape
        
        w1, b1, w2, b2 = torch.split(theta, [n_features*h_units, h_units, h_units, 1])
        h = torch.tanh(x @ w1.view(n_features,h_units) + b1)
        out = h @ w2.view(h_units, 1) + b2
        return out

def fitness_function(x, h_units=32):

    def obj(pop):
        fitnesses = []
        for theta in pop:
            
            logits = MLP(theta, x, h_units)
            
            fitness = loss(logits, y)
            fitnesses.append(fitness)
        return torch.stack(fitnesses)
    return obj

In [8]:
n_classes  = len(y.unique())
n_features = X.shape[-1]
h_units    = 128

In [9]:
n_weights = n_features*h_units + h_units + h_units + 1

In [10]:
obj_function = fitness_function(X, h_units=h_units)

In [11]:
lower_bound = [-10]*n_weights
upper_bound = [10]*n_weights

In [17]:
pop_size = 100
max_evals = pop_size*30

# Adam

In [18]:
n_epochs = max_evals

lb = torch.tensor(lower_bound, device=device)
ub = torch.tensor(upper_bound, device=device)

mean = 0.5 * (lb + ub)          # centre of the box
std  = 0.5 * (ub - lb) / 3.0    # 3-σ rule  ⇒  99.7 % inside bounds
weights = mean + std * torch.randn(1, n_weights, device=device)
weights = torch.max(torch.min(weights, ub), lb)

weights = torch.nn.Parameter(weights.squeeze(0))
optimizer = torch.optim.Adam([weights], lr=0.001) 

In [19]:
for epoch in range(n_epochs):
    optimizer.zero_grad(set_to_none=True)
    
    logits = MLP(weights, X, h_units=h_units)
    l = loss(logits, y)
    print(f"Epoch {epoch+1:4d} | Loss = {l.item():.4f}")
    l.backward()
    optimizer.step()

Epoch    1 | Loss = 6633.0254
Epoch    2 | Loss = 6588.0649
Epoch    3 | Loss = 6543.3398
Epoch    4 | Loss = 6498.8701
Epoch    5 | Loss = 6454.6611
Epoch    6 | Loss = 6410.7070
Epoch    7 | Loss = 6367.0034
Epoch    8 | Loss = 6323.5498
Epoch    9 | Loss = 6280.3477
Epoch   10 | Loss = 6237.4106
Epoch   11 | Loss = 6194.7466
Epoch   12 | Loss = 6152.3745
Epoch   13 | Loss = 6110.3037
Epoch   14 | Loss = 6068.5518
Epoch   15 | Loss = 6027.1323
Epoch   16 | Loss = 5986.0386
Epoch   17 | Loss = 5945.2729
Epoch   18 | Loss = 5904.8164
Epoch   19 | Loss = 5864.6543
Epoch   20 | Loss = 5824.7627
Epoch   21 | Loss = 5785.1250
Epoch   22 | Loss = 5745.7256
Epoch   23 | Loss = 5706.5547
Epoch   24 | Loss = 5667.6021
Epoch   25 | Loss = 5628.8618
Epoch   26 | Loss = 5590.3262
Epoch   27 | Loss = 5551.9893
Epoch   28 | Loss = 5513.8472
Epoch   29 | Loss = 5475.8926
Epoch   30 | Loss = 5438.1182
Epoch   31 | Loss = 5400.5186
Epoch   32 | Loss = 5363.0737
Epoch   33 | Loss = 5325.7700
Epoch   34

In [20]:
algorithm = CMAES(obj_function,
                dim=n_weights,
                pop_size=pop_size,
                lower_bound=lower_bound,
                upper_bound=upper_bound,
                # initialisation="gaussian",
                device=device)

minimize(algorithm, max_evals=max_evals, verbose=True)

Generation    1 | Loss = 14.5797, best_f = 14.5797
Generation    2 | Loss = 11.8162, best_f = 11.8162
Generation    3 | Loss = 9.5197, best_f = 9.5197
Generation    4 | Loss = 12.0137, best_f = 9.5197
Generation    5 | Loss = 11.0475, best_f = 9.5197
Generation    6 | Loss = 10.9034, best_f = 9.5197
Generation    7 | Loss = 10.5548, best_f = 9.5197
Generation    8 | Loss = 9.5930, best_f = 9.5197
Generation    9 | Loss = 10.2753, best_f = 9.5197
Generation   10 | Loss = 12.8044, best_f = 9.5197
Generation   11 | Loss = 9.8238, best_f = 9.5197
Generation   12 | Loss = 9.2830, best_f = 9.2830
Generation   13 | Loss = 10.4052, best_f = 9.2830
Generation   14 | Loss = 11.1822, best_f = 9.2830
Generation   15 | Loss = 11.9748, best_f = 9.2830
Generation   16 | Loss = 8.5819, best_f = 8.5819
Generation   17 | Loss = 10.9920, best_f = 8.5819
Generation   18 | Loss = 10.3923, best_f = 8.5819
Generation   19 | Loss = 11.0599, best_f = 8.5819
Generation   20 | Loss = 9.9534, best_f = 8.5819
Gene

In [16]:
from fstpso import FuzzyPSO

In [ ]:
def fitness_function_fst_pso(x, h_units=32):

    def obj(ind):
        theta = torch.tensor(ind, dtype=torch.float32).to(device)
        logits = MLP(theta, x, h_units)    
        fitness = loss(logits, y)
        return fitness.cpu().item()
    return obj

In [ ]:
obj_function_fst_pso = fitness_function_fst_pso(X, h_units=h_units)

In [ ]:
FP = FuzzyPSO()
FP.set_search_space(list(zip(lower_bound, upper_bound)))
FP.set_swarm_size(pop_size)
FP.set_fitness(obj_function_fst_pso)
FP.max_evaluations = max_evals

sigma  = 0.5 * (upper_bound[0] - lower_bound[0]) / 3.0


result = FP.solve_with_fstpso(creation_method={'name':"normal", "sigma":sigma},)